# Text-to-SVG Inference — Kaggle Submission

**Competition:** DL Spring 2026 — Text-to-SVG Generation  
**Model:** Qwen2.5-7B-Instruct + LoRA (fine-tuned on competition train set)  
**Output:** `submission.csv` with columns `id, svg`

**Kaggle dataset setup (before submitting):**
1. Upload your LoRA adapter folder to a Kaggle dataset named `svg-lora-model`
2. Upload the base model (or use HuggingFace cached) to `svg-base-model`
3. This notebook loads from `/kaggle/input/svg-lora-model/`

In [ ]:
# Install dependencies — pinned to exact versions used during training
%pip install -q unsloth==2026.3.8 transformers==5.3.0 peft==0.18.1 \
    bitsandbytes==0.49.2 accelerate==1.13.0 trl==0.24.0 \
    cairosvg lxml

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_MODEL_PATH = "/kaggle/input/svg-base-model"          # Qwen2.5-7B-Instruct base
LORA_PATH       = "/kaggle/input/svg-lora-model/lora"     # LoRA adapter (lora/ subfolder)
TEST_CSV        = "/kaggle/input/svg-competition/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Generation config
# Greedy decoding maximises SSIM score by removing randomness.
# MAX_NEW_TOKENS=3000 covers p95 SVG length (~1760 tokens) with headroom for max (~4553 tokens).
MAX_NEW_TOKENS = 3000  # p95 SVG ≈ 1760 tokens; max ≈ 4553 tokens at 3.5 chars/token
DO_SAMPLE      = False  # greedy — best for SSIM/pixel fidelity

print("Paths configured.")

In [ ]:
# ── SVG validation constants ───────────────────────────────────────────────────
# Mirrors competition rules exactly (from student guide)
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clippath", "mask", "lineargradient",
    "radialgradient", "stop", "text", "tspan", "title",
    "desc", "style", "pattern", "marker", "filter",
}

DISALLOWED_TAGS = {
    "script", "animate", "animatetransform", "animatemotion",
    "animatecolor", "set", "foreignobject",
}

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", re.IGNORECASE)

FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
    '<rect x="0" y="0" width="256" height="256" fill="white"/>'
    '<circle cx="128" cy="128" r="80" fill="#4A90D9"/>'
    '</svg>'
)


def is_valid_svg(svg_text: str) -> bool:
    if not svg_text or len(svg_text) > 16000:
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    if not root.tag.lower().endswith("svg"):
        return False
    path_count = 0
    for elem in root.iter():
        local = (elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag).lower()
        if local in DISALLOWED_TAGS:
            return False
        if local not in ALLOWED_TAGS:
            return False
        for attr in elem.attrib:
            if attr.lower().startswith("on"):
                return False
        for attr_name in ("href", "{http://www.w3.org/1999/xlink}href"):
            val = elem.attrib.get(attr_name, "")
            if val.startswith("http") or val.startswith("//"):
                return False
        if local == "path":
            path_count += 1
    return path_count <= 256


def fix_svg(svg_text: str) -> str:
    """Add required attributes if missing."""
    if len(svg_text) > 16000:
        svg_text = svg_text[:16000]
    if 'xmlns=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    if 'width=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg width="256"', 1)
    if 'height=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg height="256"', 1)
    if 'viewBox=' not in svg_text:
        svg_text = svg_text.replace("<svg", '<svg viewBox="0 0 256 256"', 1)
    return svg_text


def extract_and_validate_svg(raw_text: str) -> tuple[str, bool]:
    """Extract SVG from model output. Returns (svg, is_valid)."""
    m = SVG_REGEX.search(raw_text)
    if not m:
        return FALLBACK_SVG, False
    svg = fix_svg(m.group(0).strip())
    if is_valid_svg(svg):
        return svg, True
    return FALLBACK_SVG, False


print("SVG validation helpers loaded.")

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────────
# Must import unsloth first before any other heavy imports
import unsloth  # noqa: F401
from unsloth import FastLanguageModel
from peft import PeftModel

# Step 1: load base model (offline — from Kaggle dataset)
print(f"Loading base model from: {BASE_MODEL_PATH}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_PATH,
    max_seq_length=8192,
    dtype=None,
    load_in_4bit=True,
)

# Step 2: apply saved LoRA adapter on top of the base model
# (PeftModel.from_pretrained merges adapter weights onto the already-loaded base)
print(f"Loading LoRA adapter from: {LORA_PATH}")
model = PeftModel.from_pretrained(model, LORA_PATH)

# Step 3: enable Unsloth inference optimisations (2x faster generation)
FastLanguageModel.for_inference(model)
print("Model ready for inference.")

In [ ]:
# ── Generation function ────────────────────────────────────────────────────────
# System prompt mirrors training — must be identical to what the model was trained with
SYSTEM_PROMPT = (
    "You are an expert SVG code generator. "
    "Given a text description, output only valid SVG markup. "
    "Rules:\n"
    "- Root element: <svg xmlns=\"http://www.w3.org/2000/svg\" width=\"256\" height=\"256\" viewBox=\"0 0 256 256\">\n"
    "- Use only these elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, "
    "defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, "
    "title, desc, style, pattern, marker, filter\n"
    "- No scripts, no animation, no event handlers, no external references\n"
    "- Maximum 16000 characters, maximum 256 path elements\n"
    "- Match the visual complexity of the description — do not over-simplify or over-elaborate\n"
    "- Output only the SVG code, nothing else"
)


def build_prompt(user_text: str) -> str:
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_text}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


def generate_svg(prompt: str) -> tuple[str, bool]:
    chat_text = build_prompt(prompt)
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,       # greedy — maximises SSIM pixel fidelity
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return extract_and_validate_svg(decoded)


# Quick smoke test
test_svg, valid = generate_svg("a simple blue circle on white background")
print(f"Smoke test — valid: {valid}")
print(test_svg[:200])

In [ ]:
# ── Run inference on test set ──────────────────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
print(f"Test samples: {len(test_df)}")
print(test_df.head(3))

rows = []
fallback_count = 0
t0 = time.time()

for i, row in test_df.iterrows():
    svg, valid = generate_svg(str(row["prompt"]))
    if not valid:
        fallback_count += 1
    rows.append({"id": row["id"], "svg": svg})

    if (i + 1) % 10 == 0:
        elapsed = (time.time() - t0) / 60
        print(f"  [{i+1}/{len(test_df)}] elapsed: {elapsed:.1f} min | fallbacks: {fallback_count}")

elapsed_total = (time.time() - t0) / 60
print(f"\nDone. Total: {elapsed_total:.1f} min | Fallbacks: {fallback_count}/{len(test_df)}")

In [ ]:
# ── Save submission ────────────────────────────────────────────────────────────
sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Saved: {SUBMISSION_PATH}")
print(f"Shape: {sub_df.shape}")
print(sub_df.head(3))

# Final validation pass — must be 0
invalid_final = sub_df["svg"].apply(lambda s: not is_valid_svg(s)).sum()
print(f"\nFinal invalid count (should be 0): {invalid_final}")